# Entrainer l'OCR-VLM maison sur Kaggle

Entraine le modele from scratch (encodeur CNN + decodeur transformer, ni CLIP ni SmolLM2) sur
des tickets synthetiques multilingues dessines a la volee.

La structure est faite pour des sessions qui coupent sans prevenir : un checkpoint resumable
est ecrit a chaque epoch, et pousse vers un Dataset Kaggle dans la foulee. Une coupure coute
donc au pire l'epoch en cours.

Avant de lancer : mettre le code en Dataset et l'attacher en Input, accelerateur sur GPU T4,
internet active, et les identifiants Kaggle dans les Secrets (jamais en dur ici).

## Config

In [ ]:
# Melange de donnees reelles.
#
# Le modele entraine uniquement sur du synthetique lit tres bien... du synthetique. Sur de
# vraies photos il s'effondre : il ne lit pas, il regenere les enseignes qu'il a memorisees.
# D'ou ce run, qui melange les vrais tickets au flux synthetique. Il reprend automatiquement
# du checkpoint de l'epoch 40.
#
# Trois datasets a attacher : le bundle de code, les checkpoints, et les donnees reelles.
TARGET      = "schema"         # les vraies photos n'ont pas de transcription rendue
N_PER_EPOCH = 4000             # moins qu'avant : le signal vient maintenant du reel
LANGUAGES   = "fr,en,es,de,it"
EPOCHS      = 50               # on reprend a 40, donc 10 epochs de plus
BATCH_SIZE  = 24
LR          = 3e-4
EMBED_DIM   = 256              # doit coller au checkpoint qu'on reprend
ENC_DEPTH   = 4
DEC_DEPTH   = 4
HEADS       = 8
MAX_LEN     = 768              # idem
DISTORT     = True
INTENSITY   = "light"          # a monter si l'ecart avec le reel persiste
NUM_WORKERS = 2
LOG_EVERY   = 200
EVAL_EVERY  = 1                # NOTE: eval here is synthetic-held-out; real metrics come from
                               # scripts/evaluate_ocr_vlm.py run locally on the pushed checkpoint
KEEP_LAST   = 3

# Les donnees reelles
REAL        = True
REAL_REPEAT = 4                # sinon les 875 vrais tickets sont noyes dans le synthetique
# Auto-locate the attached real-data dataset (any /kaggle/input dir holding raw/ + labels/), so we
# don't depend on the exact Kaggle slug. Prints what's attached if it can't find it.
import glob as _glob, os as _os
def _find_real_data():
    # recursive: Kaggle may nest inputs under /kaggle/input/datasets/<slug>/... at any depth
    for _labels in _glob.glob("/kaggle/input/**/labels", recursive=True):
        _base = _os.path.dirname(_labels)
        if _os.path.isdir(_os.path.join(_base, "raw")):
            return _base
    return None
REAL_DATA_DIR = _find_real_data()
print("REAL_DATA_DIR ->", REAL_DATA_DIR)
assert not REAL or REAL_DATA_DIR, (
    "Real-data dataset not found under /kaggle/input. Add Input -> attach the receipt-vlm-real-data "
    "dataset (the zip must extract to raw/<name> + labels/<name>).")

# Stockage durable des checkpoints
SAVE_TO_DATASET  = True
SAVE_EVERY_EPOCH = True
DATASET_NAME     = "receipt-ocr-vlm-checkpoints"   # meme slug : les versions se suivent


## Le GPU est bien la ?

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Settings -> Accelerator -> GPU T4 x2"
print(torch.cuda.get_device_name(0))

## Auth Kaggle, et le helper qui pousse les checkpoints

Lit les Secrets `KAGGLE_USERNAME` et `KAGGLE_KEY`, et definit `kaggle_save()`. Les checkpoints
sont pousses fichier par fichier, pour qu'une session suivante puisse repartir de n'importe
lequel.

In [ ]:
import json, os, subprocess
from pathlib import Path

DATASET_ID = None

def _kaggle_auth():
    global DATASET_ID
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    try:
        user = sec.get_secret("KAGGLE_USERNAME")
        key  = sec.get_secret("KAGGLE_KEY")
    except Exception as e:
        raise RuntimeError(
            "Missing Secrets. Add-ons -> Secrets -> add KAGGLE_USERNAME and KAGGLE_KEY "
            "(values from Account -> Create New API Token / kaggle.json)."
        ) from e
    kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
    (kdir / "kaggle.json").write_text(json.dumps({"username": user, "key": key}))
    os.chmod(kdir / "kaggle.json", 0o600)
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
    DATASET_ID = f"{user}/{DATASET_NAME}"

def kaggle_save(folder, msg):
    """Create-or-version a Kaggle Dataset from `folder`. No-op if SAVE_TO_DATASET is False."""
    if not SAVE_TO_DATASET:
        return
    folder = Path(folder)
    if not [p for p in folder.glob("*.pt")]:
        return
    (folder / "dataset-metadata.json").write_text(json.dumps({
        "title": DATASET_NAME, "id": DATASET_ID, "licenses": [{"name": "CC0-1.0"}],
    }))
    exists = subprocess.run(["kaggle", "datasets", "files", DATASET_ID],
                            capture_output=True, text=True).returncode == 0
    sub = "version" if exists else "create"
    cmd = ["kaggle", "datasets", sub, "-p", str(folder), "--dir-mode", "skip"]
    if exists:
        cmd += ["-m", msg]
    print(f"   [dataset] {sub}: {msg}", flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("   [dataset] WARNING push failed:", r.stderr.strip()[:300], flush=True)
    else:
        print(f"   [dataset] -> https://www.kaggle.com/datasets/{DATASET_ID}", flush=True)

if SAVE_TO_DATASET:
    _kaggle_auth()
    print("Checkpoints -> dataset:", DATASET_ID, "| every epoch:", SAVE_EVERY_EPOCH)
else:
    print("SAVE_TO_DATASET=False -- /kaggle/working only (lost on interactive timeout unless you Commit)")

## Recuperer le code depuis le bundle attache

On recopie le bundle dans `/kaggle/working`, qui est le seul endroit accessible en ecriture :
un install editable ne peut pas se faire sous `/kaggle/input`, monte en lecture seule.

In [ ]:
import glob, os, shutil, zipfile
from pathlib import Path

WORK = Path("/kaggle/working/repo")

def materialize():
    zips = glob.glob("/kaggle/input/**/receipt_vlm_colab_bundle.zip", recursive=True)
    if zips:
        WORK.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORK)
        return
    hits = glob.glob("/kaggle/input/**/vlm_training/scripts/train_ocr_vlm.py", recursive=True)
    if hits:
        dev_ocr_src = Path(hits[0]).resolve().parents[2]
        dest = WORK / "dev_ocr"
        if not dest.exists():
            shutil.copytree(dev_ocr_src, dest)
        return
    raise FileNotFoundError("Bundle not found in /kaggle/input -- did you Add Input (rebuilt bundle)?")

if not list(WORK.glob("**/vlm_training/scripts/train_ocr_vlm.py")):
    materialize()

hits = glob.glob(str(WORK / "**/vlm_training/scripts/train_ocr_vlm.py"), recursive=True)
assert hits, ("train_ocr_vlm.py not found after materialize -- your uploaded bundle predates the "
              "OCR-VLM code; rebuild it with scripts/zip_selfcontained_colab.py and re-upload.")
TRAIN_PKG = Path(hits[0]).resolve().parents[1]
DEV_OCR = TRAIN_PKG.parent
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

## Installer les dependances

In [ ]:
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")
pip("-e", str(DEV_OCR))        # receipt_ocr
pip("-e", str(TRAIN_PKG))      # receipt_vlm
print("Install OK")

## Ou vont les checkpoints

Dans `/kaggle/working/ocr_ckpts`, avec le `tokenizer.json`. Le helper plus haut les recopie
vers le Dataset au fur et a mesure.

In [ ]:
from pathlib import Path
CKPT_DIR = Path("/kaggle/working/ocr_ckpts"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoints ->", CKPT_DIR)

## Reprendre apres une coupure de session

Attacher le Dataset de checkpoints en Input, puis lancer cette cellule : elle recopie les
`ocr_vlm_epoch*.pt` et le tokenizer dans le dossier de travail. `train_ocr_vlm.py` repart tout
seul du dernier epoch, il n'y a aucun flag a changer.

In [ ]:
import glob, os, shutil
restored = []
for src in glob.glob("/kaggle/input/**/ocr_vlm_epoch*.pt", recursive=True) + \
           glob.glob("/kaggle/input/**/tokenizer.json", recursive=True):
    dest = CKPT_DIR / os.path.basename(src)
    if not dest.exists():
        shutil.copy(src, dest); restored.append(dest.name)
print(f"Restored {len(restored)} file(s):", sorted(restored)[-6:] or "nothing (fresh run)")

## Entrainer

Chaque nouveau checkpoint est pousse vers le Dataset au moment ou il est ecrit.

Ces checkpoints embarquent l'etat de l'optimiseur, donc ils sont gros. `KEEP_LAST` elague les
anciens, sinon le Dataset gonfle a vue d'oeil.

In [ ]:
import subprocess, sys, os, re, time, datetime

cmd = [sys.executable, "-u", "scripts/train_ocr_vlm.py",
       "--target", TARGET, "--checkpoint-dir", str(CKPT_DIR),
       "--n", str(N_PER_EPOCH), "--languages", LANGUAGES, "--epochs", str(EPOCHS),
       "--batch-size", str(BATCH_SIZE), "--lr", str(LR),
       "--embed-dim", str(EMBED_DIM), "--enc-depth", str(ENC_DEPTH), "--dec-depth", str(DEC_DEPTH),
       "--heads", str(HEADS), "--max-len", str(MAX_LEN), "--num-workers", str(NUM_WORKERS),
       "--log-every", str(LOG_EVERY), "--eval-every", str(EVAL_EVERY), "--keep-last", str(KEEP_LAST),
       "--distort-intensity", INTENSITY]
if DISTORT:
    cmd.append("--distort")
if REAL:
    cmd += ["--real", "--real-repeat", str(REAL_REPEAT), "--real-data-dir", REAL_DATA_DIR]
print(">>", " ".join(cmd), flush=True)

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
start = last = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    now = time.time(); gap = now - last; last = now
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts} +{int(now-start):>5}s gap{gap:4.0f}s] {line}", end="", flush=True)
    if SAVE_TO_DATASET and SAVE_EVERY_EPOCH and "Checkpoint saved" in line and "_epoch" in line:
        m = re.search(r"(ocr_vlm_epoch\d+_loss[\d.]+\.pt)", line)
        kaggle_save(CKPT_DIR, m.group(1) if m else "epoch checkpoint")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"train_ocr_vlm.py failed (exit {proc.returncode})")
if SAVE_TO_DATASET and not SAVE_EVERY_EPOCH:
    kaggle_save(CKPT_DIR, "after run")
print("Training done")


## Recuperer les fichiers

Le dernier checkpoint et le `tokenizer.json` sont durables dans le Dataset, et presents dans
`/kaggle/working/ocr_ckpts`. Les deux se telechargent ensemble : un checkpoint sans le
tokenizer avec lequel il a ete entraine ne vaut rien.

In [ ]:
from pathlib import Path
print("Files in", CKPT_DIR, ":")
for p in sorted(Path(CKPT_DIR).glob("*")):
    if p.is_file():
        print(f"  {p.name:40} {p.stat().st_size/1e6:8.1f} MB")
if SAVE_TO_DATASET and DATASET_ID:
    print("\nDurable copy -> https://www.kaggle.com/datasets/" + DATASET_ID)